# 🚕 Hướng dẫn Q-Learning trên Taxi-v3

Notebook này hướng dẫn từng bước cách xây dựng và huấn luyện Q-Learning Agent trên môi trường Taxi-v3.

## Nội dung
1. Tìm hiểu môi trường Taxi-v3
2. Xây dựng Q-Learning Agent
3. Huấn luyện Agent
4. Đánh giá kết quả
5. Trực quan hóa

## 1. Cài đặt và Import

In [ ]:
# Cài đặt thư viện (chạy lần đầu)
# !pip install gymnasium numpy matplotlib seaborn tqdm

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Cấu hình hiển thị
sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Tìm hiểu môi trường Taxi-v3

### Bản đồ
```
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
```

- Lưới 5×5 với các bức tường `|`
- 4 vị trí đặc biệt: R (đỏ), G (xanh lá), Y (vàng), B (xanh dương)
- Taxi (ô vuông vàng khi trống, xanh khi chở khách)
- Hành khách (chữ cái xanh biển)
- Đích đến (chữ cái tím)

In [ ]:
# Tạo môi trường
env = gym.make("Taxi-v3", render_mode="ansi")

# Thông tin cơ bản
print(f"Số trạng thái: {env.observation_space.n}")  # 500
print(f"Số hành động : {env.action_space.n}")        # 6
print()

# Reset và hiển thị
state, info = env.reset(seed=42)
print(f"Trạng thái ban đầu: {state}")
print(env.render())

In [ ]:
# Giải mã trạng thái
def decode_state(state):
    """Giải mã trạng thái thành (hàng taxi, cột taxi, vị trí khách, đích)"""
    taxi_row = state // 100
    remainder = state % 100
    taxi_col = remainder // 20
    remainder = remainder % 20
    passenger_loc = remainder // 4
    destination = remainder % 4
    return taxi_row, taxi_col, passenger_loc, destination

locations = {0: "R", 1: "G", 2: "Y", 3: "B", 4: "Trên xe"}
actions = {0: "South ↓", 1: "North ↑", 2: "East →", 3: "West ←", 4: "Pickup", 5: "Dropoff"}

row, col, passenger, dest = decode_state(state)
print(f"Taxi tại: ({row}, {col})")
print(f"Hành khách: {locations[passenger]}")
print(f"Đích đến: {locations[dest]}")

In [ ]:
# Thử thực hiện một vài hành động
print("=== Thử hành động ngẫu nhiên ===")
state, _ = env.reset(seed=42)

for step in range(5):
    action = env.action_space.sample()  # Chọn ngẫu nhiên
    next_state, reward, terminated, truncated, info = env.step(action)
    print(f"\nBước {step + 1}: Hành động = {actions[action]}, Reward = {reward}")
    print(env.render())
    state = next_state
    if terminated:
        print("Episode kết thúc!")
        break

env.close()

## 3. Xây dựng Q-Learning Agent

### Công thức Q-Learning

$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \right]$$

Trong đó:
- $\alpha$ (learning rate): tốc độ học
- $\gamma$ (discount factor): hệ số chiết khấu
- $\epsilon$ (epsilon): xác suất khám phá

In [ ]:
class QLearningAgent:
    """Q-Learning Agent đơn giản."""
    
    def __init__(self, n_states, n_actions, lr=0.1, gamma=0.99,
                 epsilon=1.0, epsilon_end=0.01, epsilon_decay=0.9995):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = lr              # Tốc độ học
        self.gamma = gamma        # Hệ số chiết khấu
        self.epsilon = epsilon    # Xác suất khám phá
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Q-table: bảng 500 x 6, khởi tạo = 0
        self.q_table = np.zeros((n_states, n_actions))
    
    def choose_action(self, state):
        """Chọn hành động theo epsilon-greedy."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # Khám phá
        return np.argmax(self.q_table[state])          # Khai thác
    
    def update(self, state, action, reward, next_state):
        """Cập nhật Q-value."""
        best_next = np.max(self.q_table[next_state])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.lr * td_error
    
    def decay_epsilon(self):
        """Giảm epsilon."""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

print("Agent đã được định nghĩa!")

## 4. Huấn luyện Agent

In [ ]:
# Tham số huấn luyện
N_EPISODES = 10000
SEED = 42

# Khởi tạo
np.random.seed(SEED)
env = gym.make("Taxi-v3")
agent = QLearningAgent(
    n_states=env.observation_space.n,
    n_actions=env.action_space.n,
)

# Lưu lịch sử
rewards_history = []
steps_history = []
epsilon_history = []

# Vòng lặp huấn luyện
for episode in tqdm(range(N_EPISODES), desc="Huấn luyện"):
    state, _ = env.reset(seed=SEED + episode)
    total_reward = 0
    steps = 0
    done = False
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        agent.update(state, action, reward, next_state)
        
        state = next_state
        total_reward += reward
        steps += 1
    
    agent.decay_epsilon()
    rewards_history.append(total_reward)
    steps_history.append(steps)
    epsilon_history.append(agent.epsilon)

env.close()

# Kết quả
last_1000 = rewards_history[-1000:]
print(f"\nReward trung bình (1000 ep cuối): {np.mean(last_1000):.2f}")
print(f"Steps trung bình (1000 ep cuối): {np.mean(steps_history[-1000:]):.1f}")
print(f"Epsilon cuối: {agent.epsilon:.4f}")

## 5. Trực quan hóa kết quả

In [ ]:
def moving_average(values, window=100):
    """Tính trung bình trượt."""
    cumsum = np.cumsum(np.insert(values, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / window

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Reward theo episode
ax = axes[0, 0]
ax.plot(rewards_history, alpha=0.15, color="steelblue")
reward_ma = moving_average(np.array(rewards_history))
ax.plot(range(100, len(rewards_history) + 1), reward_ma, color="darkblue", linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.set_title("Reward theo Episode")
ax.axhline(y=0, color="red", linestyle="--", alpha=0.3)

# 2. Steps theo episode
ax = axes[0, 1]
ax.plot(steps_history, alpha=0.15, color="coral")
steps_ma = moving_average(np.array(steps_history))
ax.plot(range(100, len(steps_history) + 1), steps_ma, color="darkred", linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Số bước")
ax.set_title("Số bước theo Episode")

# 3. Epsilon
ax = axes[1, 0]
ax.plot(epsilon_history, color="green", linewidth=2)
ax.fill_between(range(len(epsilon_history)), epsilon_history, alpha=0.2, color="green")
ax.set_xlabel("Episode")
ax.set_ylabel("Epsilon")
ax.set_title("Epsilon Decay")

# 4. Phân phối reward
ax = axes[1, 1]
half = len(rewards_history) // 2
ax.hist(rewards_history[:half], bins=30, alpha=0.6, color="salmon", label="Nửa đầu", density=True)
ax.hist(rewards_history[half:], bins=30, alpha=0.6, color="steelblue", label="Nửa sau", density=True)
ax.set_xlabel("Reward")
ax.set_ylabel("Mật độ")
ax.set_title("Phân phối Reward")
ax.legend()

plt.tight_layout()
plt.show()

## 6. Đánh giá Agent đã huấn luyện

In [ ]:
# Đánh giá trên 100 episode (không khám phá)
env = gym.make("Taxi-v3", render_mode="ansi")
agent.epsilon = 0.0  # Tắt exploration

eval_rewards = []
eval_steps = []

for ep in range(100):
    state, _ = env.reset(seed=ep)
    total_reward = 0
    steps = 0
    done = False
    
    while not done:
        action = np.argmax(agent.q_table[state])  # Greedy
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
    
    eval_rewards.append(total_reward)
    eval_steps.append(steps)

print(f"Reward trung bình: {np.mean(eval_rewards):.2f} ± {np.std(eval_rewards):.2f}")
print(f"Steps trung bình : {np.mean(eval_steps):.1f} ± {np.std(eval_steps):.1f}")
print(f"Tỷ lệ thành công : {sum(1 for r in eval_rewards if r > 0) / len(eval_rewards) * 100:.0f}%")

In [ ]:
# Demo 1 episode
state, _ = env.reset(seed=42)
print("=== DEMO: Agent chơi 1 episode ===")
print(f"\nTrạng thái ban đầu (state={state}):")
print(env.render())

total_reward = 0
step = 0
done = False

while not done:
    action = np.argmax(agent.q_table[state])
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward
    step += 1
    print(f"\nBước {step}: {actions[action]} | Reward: {reward} | Tổng: {total_reward}")
    print(env.render())

print(f"\nKết quả: {total_reward} reward trong {step} bước")
env.close()

## 7. Phân tích Q-Table

In [ ]:
# Heatmap Q-table (50 trạng thái tiêu biểu)
q_range = np.ptp(agent.q_table, axis=1)
top_states = np.argsort(q_range)[-50:]

plt.figure(figsize=(10, 12))
action_labels = ["↓ South", "↑ North", "→ East", "← West", "Pickup", "Dropoff"]
sns.heatmap(
    agent.q_table[top_states],
    xticklabels=action_labels,
    yticklabels=[str(s) for s in top_states],
    cmap="RdYlGn",
    center=0,
)
plt.xlabel("Hành động")
plt.ylabel("Trạng thái")
plt.title("Q-Table Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Xem Q-value cho một trạng thái cụ thể
example_state = 328  # Taxi (3,2), khách tại Y, đích R
row, col, passenger, dest = decode_state(example_state)

print(f"Trạng thái {example_state}:")
print(f"  Taxi: ({row}, {col})")
print(f"  Khách: {locations[passenger]}")
print(f"  Đích: {locations[dest]}")
print(f"\nQ-values:")
for a in range(6):
    q_val = agent.q_table[example_state, a]
    best = " ← TỐT NHẤT" if a == np.argmax(agent.q_table[example_state]) else ""
    print(f"  {actions[a]:>12s}: {q_val:>8.3f}{best}")

## 8. Tổng kết

### Những gì đã học:
1. **Môi trường RL**: Cách tương tác với Gymnasium (reset, step, render)
2. **Q-Table**: Bảng lưu giá trị hành động cho mỗi trạng thái
3. **ε-greedy**: Cân bằng giữa khám phá và khai thác
4. **TD Learning**: Cập nhật Q-value dựa trên sai số temporal difference
5. **Epsilon Decay**: Giảm dần exploration để hội tụ

### Thử nghiệm thêm:
- Thay đổi learning rate (α) và xem ảnh hưởng
- Thay đổi discount factor (γ) và so sánh
- So sánh với SARSA (chạy `python compare_algorithms.py`)
- Thử các giá trị epsilon_decay khác nhau